In [ ]:
import torch
import torch.nn as nn 
import pandas as pd
from tqdm import tqdm
import csv
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingWarmRestarts
from torch.cuda.amp import autocast, GradScaler

from transformers import CLIPModel, CLIPProcessor
from peft import LoraConfig, get_peft_model, TaskType

/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class PriceDataset(Dataset):
    def __init__(self, img_path, image_paths, item_names, bullet_points, 
                 product_descriptions, values, units, prices, processor, max_length=77):
        self.img_dir = img_path
        self.image_paths = image_paths
        self.item_names = item_names
        self.bullet_points = bullet_points
        self.product_descriptions = product_descriptions
        self.values = values
        self.units = units
        self.prices = torch.tensor(prices, dtype=torch.float32)
        self.processor = processor
        self.max_length = max_length
    
    def __len__(self):
        return len(self.image_paths)
    
    def _concatenate_text(self, idx):
        """
        Concatenate all text fields into one string.
        Prioritizes most important information first (for token limit).
        """
        text_parts = []
        
        # Item Name (most important)
        if pd.notna(self.item_names[idx]) and str(self.item_names[idx]).strip():
            text_parts.append(str(self.item_names[idx]).strip())
        
        # Quantity (concise and important for pricing)
        if pd.notna(self.values[idx]) and pd.notna(self.units[idx]):
            text_parts.append(f"{self.values[idx]} {self.units[idx]}")
        
        # Bullet Points (key features)
        if pd.notna(self.bullet_points[idx]) and str(self.bullet_points[idx]).strip():
            text_parts.append(str(self.bullet_points[idx]).strip())
        
        # Product Description (detailed, might be truncated)
        if pd.notna(self.product_descriptions[idx]) and str(self.product_descriptions[idx]).strip():
            text_parts.append(str(self.product_descriptions[idx]).strip())
        
        # Join with separator that CLIP handles well
        full_text = ". ".join(text_parts)
        
        # Handle empty text case
        if not full_text.strip():
            full_text = "product"  # Fallback text
        
        return full_text
    
    def __getitem__(self, idx):
        """
        Returns a single sample with image and text features.
        """
        # Load image
        img_filename = f"{self.image_paths[idx]}.jpg"
        img_path = os.path.join(self.img_dir, img_filename)
        
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a blank image as fallback
            img = Image.new('RGB', (224, 224), color='white')
        
        # Concatenate text fields
        full_text = self._concatenate_text(idx)
        
        # Process with CLIP processor
        inputs = self.processor(
            text=full_text,
            images=img,
            return_tensors="pt",
            padding="max_length",  # Consistent padding
            truncation=True,
            max_length=self.max_length
        )
        
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'price': self.prices[idx]
        }

In [ ]:
class SMAPELoss(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps
    
    def forward(self, pred, target):
        # Ensure inputs are valid
        pred = torch.clamp(pred, min=-1e6, max=1e6)  # Prevent extreme values
        target = torch.clamp(target, min=-1e6, max=1e6)
        
        numerator = torch.abs(pred - target)
        denominator = (torch.abs(pred) + torch.abs(target) + self.eps)
        
        # Prevent division by zero
        denominator = torch.clamp(denominator, min=self.eps)
        
        smape = 2.0 * numerator / denominator
        
        # Check for NaN/inf before returning
        smape = torch.where(torch.isnan(smape) | torch.isinf(smape), 
                           torch.zeros_like(smape), smape)
        
        return torch.mean(smape) * 100

In [ ]:
class CLIPPricePredictor(nn.Module):
    def __init__(self, model_name="openai/clip-vit-base-patch32"):
        super().__init__()
        # Load CLIP model
        self.clip = CLIPModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="cuda"
        )
        
        # Freeze CLIP parameters
        for param in self.clip.parameters():
            param.requires_grad = False
        
        # Get CLIP projection dimension (typically 512 for base models)
        clip_dim = self.clip.config.projection_dim
        model_device = next(self.clip.parameters()).device
        
        # Regression head for multimodal features
        self.regressor = nn.Sequential(
            nn.LayerNorm(clip_dim),
            nn.Linear(clip_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )
        
        self._init_weights()
        self.regressor.to(model_device, dtype=torch.float32)
    
    def _init_weights(self):
        for layer in self.regressor:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight, gain=0.01)
                nn.init.zeros_(layer.bias)
    
    @torch.no_grad()
    def extract_features(self, pixel_values, input_ids, attention_mask):
        """
        Extract frozen multimodal embeddings from CLIP.
        Combines image and text features in the shared embedding space.
        """
        self.clip.eval()
        
        # Get CLIP outputs
        outputs = self.clip(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Extract normalized embeddings from CLIP's projection space
        image_embeds = outputs.image_embeds  # [batch_size, projection_dim]
        text_embeds = outputs.text_embeds    # [batch_size, projection_dim]
        
        # Combine image and text features (average fusion)
        # You can also try concatenation or other fusion strategies
        multimodal_features = (image_embeds + text_embeds) / 2
        
        # Clamp for stability
        return torch.clamp(multimodal_features.float(), min=-5, max=5)
    
    def forward(self, pixel_values, input_ids, attention_mask=None):
        """
        Forward pass for price prediction.
        
        Args:
            pixel_values: Preprocessed images [batch_size, 3, H, W]
            input_ids: Tokenized text [batch_size, seq_len]
            attention_mask: Attention mask for text [batch_size, seq_len]
        
        Returns:
            price_pred: Predicted prices [batch_size]
        """
        with torch.no_grad():
            features = self.extract_features(pixel_values, input_ids, attention_mask)
        
        price_pred = self.regressor(features)
        return price_pred.squeeze(-1)

In [6]:
def apply_qlora(model, r=16, alpha=32):
    lora_config = LoraConfig(
        r=r,
        lora_alpha=alpha,
        target_modules=[
            "query", "key", "value",
            "attention.self.query",
            "attention.self.key", 
            "attention.self.value",
            "crossattention.self.query",
            "crossattention.self.key",
            "crossattention.self.value"
        ],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.FEATURE_EXTRACTION,
    )
    
    model.blip2.qformer = get_peft_model(model.blip2.qformer, lora_config)
    
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    
    return model

In [ ]:
def train(model, train_loader, val_loader, epochs=10, lr=1e-4, device='cuda', log_file='training_log.csv'):
    model.to(device)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=1e-2
    )
    
    # Use MSE or Huber loss for regression
    criterion = nn.MSELoss()
    
    # Simple cosine annealing scheduler
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
    
    # Setup logging
    if not os.path.exists(log_file):
        with open(log_file, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Epoch', 'Train_Loss', 'Val_Loss', 'LR'])
    
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        
        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
        for batch in train_pbar:
            # Move data to device with correct dtypes
            pixel_values = batch['pixel_values'].to(device, dtype=torch.float16)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            prices = batch['price'].to(device, dtype=torch.float32)
            
            # Forward pass
            optimizer.zero_grad()
            predictions = model(pixel_values, input_ids, attention_mask)
            
            # Compute loss
            loss = criterion(predictions, prices)
            
            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'lr': f'{optimizer.param_groups[0]["lr"]:.2e}'
            })
        
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        
        val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]', leave=False)
        with torch.no_grad():
            for batch in val_pbar:
                pixel_values = batch['pixel_values'].to(device, dtype=torch.float16)
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                prices = batch['price'].to(device, dtype=torch.float32)
                
                predictions = model(pixel_values, input_ids, attention_mask)
                loss = criterion(predictions, prices)
                
                val_loss += loss.item()
                val_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_val_loss = val_loss / len(val_loader)
        
        # Update learning rate
        scheduler.step()
        
        # Log metrics
        current_lr = optimizer.param_groups[0]['lr']
        with open(log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([epoch + 1, avg_train_loss, avg_val_loss, current_lr])
        
        print(f'Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, LR: {current_lr:.2e}')
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_model.pth')
            print(f'✓ Saved best model with val_loss: {best_val_loss:.4f}')
        
        # Clear cache periodically
        if (epoch + 1) % 5 == 0:
            torch.cuda.empty_cache()
    
    print(f'\nTraining complete! Best validation loss: {best_val_loss:.4f}')
    return model

In [ ]:
# MODEL = "Salesforce/blip2-opt-2.7b"
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
TRAIN_IMG_DIR = '../input/amazon-train-set-part1/train_part1/train_part1'
VAL_IMG_DIR = '../input/amazon-train-set-part1/val_part1/val_part1'
train_df = pd.read_csv('../input/amazon-train-set-part1/train_part1.csv')
val_df = pd.read_csv('../input/amazon-train-set-part1/val_part1.csv') 

train_dl = DataLoader(
    PriceDataset(
        img_path = TRAIN_IMG_DIR,
        image_paths=train_df['sample_id'].values,
        item_names=train_df['Item Name'].values,
        bullet_points=train_df['Bullet Points'].values,
        product_descriptions=train_df['Product Description'].values,
        values=train_df['Value'].values,
        units=train_df['Unit'].values,
        prices=train_df['log_price'].values,  
        processor=processor
    ),
    batch_size=4,
    shuffle=True,
    num_workers=4
)
val_dl = DataLoader(
    PriceDataset(
        img_path = VAL_IMG_DIR,
        image_paths=val_df['sample_id'].values,
        item_names=val_df['Item Name'].values,
        bullet_points=val_df['Bullet Points'].values,
        product_descriptions=val_df['Product Description'].values,
        values=val_df['Value'].values,
        units=val_df['Unit'].values,
        prices=val_df['log_price'].values,  
        processor=processor
    ),
    batch_size=4,
    shuffle=False,
    num_workers=4
)    

model = CLIPPricePredictor()
# model = apply_qlora(model, r=16, alpha=32)
    
model = train(model, train_dl, val_dl, epochs=10, lr=1.09e-4)
torch.save({
    'regressor': model.regressor.state_dict(),
    'model_name': model.clip.config._name_or_path,  # CLIP model identifier
    'clip_dim': model.clip.config.projection_dim,
}, 'model.pth')
    
print("Done!")

Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.34s/it]
/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/accelerate/utils/modeling.py:1614: UserWarning: The following device_map keys do not match any submodules in the model: ['query_tokens']
  warnings.warn(
Some parameters are on the meta device because they were offloaded to the cpu.


Trainable: 25,088,001 / 3,898,628,097 (0.64%)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/torch/utils/data/_utils/fetch.py", line 55, in fetch
    return self.collate_fn(data)
           ~~~~~~~~~~~~~~~^^^^^^
  File "/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/torch/utils/data/_utils/collate.py", line 398, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
  File "/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/torch/utils/data/_utils/collate.py", line 172, in collate
    key: collate(
         ~~~~~~~^
        [d[key] for d in batch], collate_fn_map=collate_fn_map
        ^^^^^^^^^^^^^^^^^^^^^^^^^